In [ ]:
import pandas as pd

# 指定文件路径
file_path = r'data/raw/executives/TMT_FIGUREINFO.csv'

# 使用 pandas 读取 CSV 文件
df = pd.read_csv(file_path,error_bad_lines=False)

# 显示数据的前几行，确认数据已成功读取
print(df.head())

In [ ]:
import pandas as pd

# 指定文件路径
file_path = r'data/work/unique_Stkcd2.csv'

# 使用 pandas 读取 CSV 文件
unique_Stkcd = pd.read_csv(file_path,error_bad_lines=False)

# 显示数据的前几行，确认数据已成功读取
print(unique_Stkcd.head())

In [ ]:
# 使用 isin() 方法筛选出 df_combined 中 Stkcd 列匹配 unique_Stkcd 的数据
df = df[df['Stkcd'].isin(unique_Stkcd['Stkcd'])]

# 显示筛选后的数据
print(df)

In [ ]:
columns_to_drop = ['Nationality', 'NatAreaCode', 'BirAreaCode', 'Gender', 'PaidSign', 'OveseaBack', 'Academic']
df_cleaned = df.drop(columns=columns_to_drop)

In [ ]:
df_cleaned

In [ ]:
#仅筛选出高管
df=df_cleaned[df_cleaned['IsMTMT']==1]
df

# 注意数据库表明日期是统计截止日期，且每年统计1次，而我的第一个数据集是2022年1月到2023年12月31日，那么我就找2021年12月的，其会参与到2022年的；2022年12月的，其会参加到2023年12月的

In [ ]:
df[df['Stkcd']==1]

In [ ]:
df1 = df[df['Reptdt'].isin(['2021-12-31', '2022-12-31'])]

In [ ]:
df1

In [ ]:
len(df['Stkcd'].unique())

# 第一步，筛选出同时存在2年的高管

对于每一个股票（Stkcd），Reptdt都有两个日期'2015-12-31', '2016-12-31'，而每一个时期都有很多的人名Name，假如某个人名Name在这个股票的这两个时期都出现了就筛选出来

In [ ]:
result = df1.groupby(['Stkcd', 'Name']).filter(lambda x: x['Reptdt'].nunique() == 2)
result

In [ ]:
result[result['Stkcd']==4]

# 仅保留两个的前一个

In [ ]:
result_filtered = result[result['Reptdt'] == '2021-12-31']

# 查询同一个高管在多家公司担任

In [ ]:
# 统计 'Name' 列每个名字出现的次数
name_counts = result_filtered['Name'].value_counts()

# 输出名字出现的次数
print(name_counts)

In [ ]:
k=0
for i in range(len(name_counts)):
    if name_counts[i]>=2:
        print(name_counts[i])
        k+=1
print(k)

In [ ]:
result_filtered

# 筛选出在多个股票公司中任职的高管信息，并生成高管与股票的关系表（即高管同时在多个股票公司任职的股票对关系）

In [ ]:
# 假设 result_filtered 是已经加载的 pandas DataFrame，直接进行处理

# 步骤 1：筛选出同时在多个股票公司中任职的高管
executive_counts = result_filtered.groupby('Name')['Stkcd'].nunique()
executives_in_multiple_stocks = executive_counts[executive_counts > 1].index.tolist()

# 筛选出这些高管的相关行
multiple_stock_executives_df = result_filtered[result_filtered['Name'].isin(executives_in_multiple_stocks)]

# 步骤 2：创建高管和股票之间的关系表，形成【高管，股票A，股票B】的结构
relationship_rows = []

for name in executives_in_multiple_stocks:
    stocks = multiple_stock_executives_df[multiple_stock_executives_df['Name'] == name]['Stkcd'].unique()
    for i in range(len(stocks)):
        for j in range(i+1, len(stocks)):
            relationship_rows.append([name, stocks[i], stocks[j]])

relationship_df = pd.DataFrame(relationship_rows, columns=['Executive', 'StockA', 'StockB'])

# 返回两个结果表格
multiple_stock_executives_df, relationship_df

In [ ]:
relationship_df

# relationship_df 进一步对其进行操作，如果【高管，股票A，股票B】和【高管，股票B，股票A】则认为这两个是一样的，只保留一个

In [ ]:
# 对股票的 A 和 B 列排序，使得 [A, B] 和 [B, A] 被认为是相同的
relationship_df['StockA'], relationship_df['StockB'] = relationship_df[['StockA', 'StockB']].min(axis=1), relationship_df[['StockA', 'StockB']].max(axis=1)

# 然后我们去重
relationship_df = relationship_df.drop_duplicates(subset=['Executive', 'StockA', 'StockB'])

# 返回去重后的结果
relationship_df 

# 对于relationship_df，看看StockA 和 StockB两列中总的每个股票的出现次数,并找到大于出现1次的所有股票序号

至此这一步讲讲所有的过程，首先我想看看这个总表高管在多个公司任职的情况，至少有2个才可形成边，结果发现有820个公司都存在一个高管面临管理同时2个公司及以上的情况，这会导致公司数量太多，而邻接矩阵很稀疏的问题，因此我就做了这一步，目的是找到更紧密的邻接矩阵，同时缩减公司的数量

In [ ]:
# 将 StockA 和 StockB 两列合并
all_stocks = pd.concat([relationship_df['StockA'], relationship_df['StockB']])

# 计算每个股票的出现次数
stock_counts = all_stocks.value_counts()

# 找到出现次数大于 3 的所有股票序号
stocks_more_than_two = stock_counts[stock_counts > 1].index.tolist()

# 返回结果
stocks_more_than_two

In [ ]:
len(stocks_more_than_two)

In [ ]:
import numpy as np
np.array(stocks_more_than_two)

# !!!!!!现在将这个序号去日收率数据看看有没有缺失值，进一步从这455个公司筛选出来新的list¶

In [ ]:
stock_list =[34, 403, 513, 526, 568, 581, 625, 636, 661, 681, 736, 821, 858, 869, 895, 915, 963, 977, 988, 997, 999, 1206, 1210, 1212, 1219, 2009, 2025, 2030, 2032, 2048, 2074, 2126, 2128, 2151, 2156, 2180, 2214, 2223, 2236, 2250, 2253, 2258, 2268, 2271, 2281, 2287, 2291, 2294, 2415, 2430, 2432, 2462, 2466, 2475, 2484, 2508, 2541, 2549, 2555, 2568, 2585, 2594, 2595, 2597, 2601, 2648, 2653, 2657, 2690, 2705, 2733, 2737, 2867, 2868, 2873, 2878, 2884, 2886, 2895, 2906, 2907, 2912, 2915, 2923, 2932, 2935, 2941, 2949, 2968, 2971, 2972, 2975, 2977, 2978, 2987, 2988, 2990, 3008, 3009, 3020, 3021, 3022, 3023, 3029, 3030, 3033, 3038, 600031, 600048, 600071, 600089, 600141, 600148, 600160, 600188, 600196, 600197, 600285, 600295, 600323, 600329, 600346, 600389, 600418, 600419, 600426, 600452, 600456, 600476, 600498, 600499, 600520, 600521, 600529, 600557, 600559, 600563, 600566, 600570, 600588, 600612, 600645, 600699, 600702, 600732, 600741, 600763, 600773, 600809, 600859, 600862, 600872, 600875, 600893, 600933, 600988, 600990, 600993, 600999, 601012, 601021, 601100, 601126, 601156, 601211, 601233, 601595, 601601, 601607, 601615, 601677, 601688, 601689, 601698, 601838, 601858, 603319, 603324, 603332, 603355, 603360, 603363, 603383, 603393, 603439, 603456, 603517, 603529, 603533, 603565, 603566, 603567, 603585, 603587, 603595, 603611, 603613, 603630, 603658, 603659, 603693, 603712, 603713, 603721, 603739, 603776, 603786, 603787, 603801, 603825, 603848, 603860, 603871, 603881, 603883, 603885, 603888, 603893, 603896, 603908, 603916, 603919, 603927, 603987, 603992, 605122, 605123, 605198, 605286, 605289, 605296, 605338, 605366, 605567]

# 现在对于这个新的股票序号，反过来筛选relationship_df的数据，即如果relationship_df每一行的股票A和股票B均在这个股票序号，则保留否则删除¶

# 从这一步开始正式构建邻接矩阵，就从上一步筛选的这181个公司

In [ ]:
# 反向筛选，保留 relationship_df 中 StockA 和 StockB 都在 stocks_more_than_two 列表中的行
filtered_relationship_df = relationship_df[
    relationship_df['StockA'].isin(stocks_more_than_two) & 
    relationship_df['StockB'].isin(stocks_more_than_two)
]

# 返回筛选后的表格
filtered_relationship_df



In [ ]:
# 统计 filtered_relationship_df 中独特的高管数量
unique_executives_count = filtered_relationship_df['Executive'].nunique()

# 返回高管的数量
unique_executives_count

# 可以看出这234个公司共构成1653条边，里面有279个高管¶

In [ ]:
filtered_relationship_df['Executive'].unique()
#len(filtered_relationship_df['Executive'].unique())

In [ ]:
# 使用 pd.melt() 将 Stock1 和 Stock2 合并为一列
stock_per_exe = pd.melt(filtered_relationship_df, id_vars=['Executive'], value_vars=['StockA', 'StockB'], 
                                var_name='StockType', value_name='Stock')

# 去重，确保每个股东每个股票只出现一次
stock_per_exe = stock_per_exe.drop_duplicates(subset=['Executive', 'Stock'])

# 按股东分组，列出他们持有的股票
stocks_by_exe = stock_per_exe.groupby('Executive')['Stock'].apply(list)

# 展示结果
stocks_by_exe

In [ ]:
import pandas as pd

# 假设 stock_list 和 stocks_by_shareholder 已经定义
# stock_list 是您之前给出的股票列表
# stocks_by_shareholder 是每个股东持有的股票列表
stock_list =[34, 403, 513, 526, 568, 581, 625, 636, 661, 681, 736, 821, 858, 869, 895, 915, 963, 977, 988, 997, 999, 1206, 1210, 1212, 1219, 2009, 2025, 2030, 2032, 2048, 2074, 2126, 2128, 2151, 2156, 2180, 2214, 2223, 2236, 2250, 2253, 2258, 2268, 2271, 2281, 2287, 2291, 2294, 2415, 2430, 2432, 2462, 2466, 2475, 2484, 2508, 2541, 2549, 2555, 2568, 2585, 2594, 2595, 2597, 2601, 2648, 2653, 2657, 2690, 2705, 2733, 2737, 2867, 2868, 2873, 2878, 2884, 2886, 2895, 2906, 2907, 2912, 2915, 2923, 2932, 2935, 2941, 2949, 2968, 2971, 2972, 2975, 2977, 2978, 2987, 2988, 2990, 3008, 3009, 3020, 3021, 3022, 3023, 3029, 3030, 3033, 3038, 600031, 600048, 600071, 600089, 600141, 600148, 600160, 600188, 600196, 600197, 600285, 600295, 600323, 600329, 600346, 600389, 600418, 600419, 600426, 600452, 600456, 600476, 600498, 600499, 600520, 600521, 600529, 600557, 600559, 600563, 600566, 600570, 600588, 600612, 600645, 600699, 600702, 600732, 600741, 600763, 600773, 600809, 600859, 600862, 600872, 600875, 600893, 600933, 600988, 600990, 600993, 600999, 601012, 601021, 601100, 601126, 601156, 601211, 601233, 601595, 601601, 601607, 601615, 601677, 601688, 601689, 601698, 601838, 601858, 603319, 603324, 603332, 603355, 603360, 603363, 603383, 603393, 603439, 603456, 603517, 603529, 603533, 603565, 603566, 603567, 603585, 603587, 603595, 603611, 603613, 603630, 603658, 603659, 603693, 603712, 603713, 603721, 603739, 603776, 603786, 603787, 603801, 603825, 603848, 603860, 603871, 603881, 603883, 603885, 603888, 603893, 603896, 603908, 603916, 603919, 603927, 603987, 603992, 605122, 605123, 605198, 605286, 605289, 605296, 605338, 605366, 605567]
# 提取所有股东（列名）
unique_exe = stocks_by_exe.index

# 创建一个全零矩阵，行是股票，列是股东
matrix = pd.DataFrame(0, index=stock_list, columns=unique_exe)

# 遍历 stocks_by_shareholder，为每个股东持有的股票在矩阵中填 1
for exe, stocks in stocks_by_exe.items():
    for stock in stocks:
        if stock in stock_list:
            matrix.at[stock, exe] = 1

# 返回最终构造的矩阵
matrix

In [ ]:
(matrix == 1).sum().sum()

# 这个矩阵用于构造股东的初始特征

In [ ]:
# 将矩阵保存为 CSV 文件，不包括行和列标签
output_path = r'ComNC_main/CAS2\matrix.csv'
matrix.to_csv(output_path, index=False, header=False)

# 现在构建第二张表，这是总的关系表

In [ ]:
# 构造矩阵的行和列，行和列均包含股票以及股东
matrix_index = list(stock_list) + list(unique_exe)
matrix = pd.DataFrame(0, index=matrix_index, columns=matrix_index)

# Step 1: 处理 stock_list 对应 stock_list 的关系
for _, row in filtered_relationship_df.iterrows():
    stockA, stockB, exe = row['StockA'], row['StockB'], row['Executive']
    
    # 如果 stockA 和 stockB 都在 stock_list 中，构造双向关系
    if stockA in stock_list and stockB in stock_list:
        matrix.at[stockA, stockB] = 1
        matrix.at[stockB, stockA] = 1

# Step 2: 处理 stock_list 对应 unique_shareholders 的关系
for _, row in filtered_relationship_df.iterrows():
    stockA, stockB, exe = row['StockA'], row['StockB'], row['Executive']
    
    # 如果 stockA 在 stock_list 且 shareholder 在 unique_shareholders 中，填充 1
    if stockA in stock_list and exe in unique_exe:
        matrix.at[stockA, exe] = 1
    
    # 如果 stockB 在 stock_list 且 shareholder 在 unique_shareholders 中，填充 1
    if stockB in stock_list and exe in unique_exe:
        matrix.at[stockB, exe] = 1

# 返回构造的矩阵
matrix

In [ ]:
(matrix == 1).sum().sum()

In [ ]:
# 创建两个列表用于存储行和列的序号
row_indices = []
column_indices = []

# 遍历矩阵，检查每个元素是否为 1，如果是 1 则记录其行和列的序号
for row_idx, (row_name, row) in enumerate(matrix.iterrows()):  # 使用 enumerate 获取行的序号
    for col_idx, (col_name, value) in enumerate(row.items()):  # 使用 enumerate 获取列的序号
        if value == 1:
            row_indices.append(row_idx )  # 行序号
            column_indices.append(col_idx )  # 列序号

# 将行序号和列序号组合成一个大列表
result = [row_indices, column_indices]

# 输出结果
result

In [ ]:
import torch

# 假设 result 是一个包含行索引和列索引的列表
# 将 result 列表转换为张量
result_tensor = torch.tensor(result, dtype=torch.int64)

# 输出转换后的张量
print(result_tensor)

In [ ]:
len(result_tensor[0])

In [ ]:
# import torch
# import pandas as pd

# # 假设 result 是一个包含行索引和列索引的列表
# # 将 result 列表转换为张量
# result_tensor = torch.tensor(result, dtype=torch.int64)

# # 将张量转换为 DataFrame
# df000 = pd.DataFrame(result_tensor.numpy().T, columns=['Row_Index', 'Col_Index'])

# # 保存为 CSV 文件
# output_path = r'ComNC_main/CAS2\firm_executives.csv'
# df000.to_csv(output_path, index=False)

# print(f"保存成功: {output_path}")